# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their @id, and fields

record_set_ids = []

for rset in dataset.metadata.record_sets:
    print(f"RecordSet name: {rset.name}; @id: {rset.id}")
    record_set_ids.append(rset.id)
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name}: @id={field.id}, data_type={field.data_type}")
    print()

# Choose one record set for demonstration (use the first one)
if record_set_ids:
    example_record_set_id = record_set_ids[0]
else:
    example_record_set_id = None

print(f"Example RecordSet for data extraction: {example_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into a DataFrame

dataframes = {}

for rset in dataset.metadata.record_sets:
    try:
        records = list(dataset.records(record_set=rset.id))
        df = pd.DataFrame(records)
        dataframes[rset.id] = df
        print(f"Loaded {len(df)} records for RecordSet {rset.id}")
    except Exception as e:
        print(f"Could not load records for RecordSet {rset.id}: {e}")

# Print DataFrame columns for the example record set
if example_record_set_id in dataframes:
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No data available for selected record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
from IPython.display import display

# Identify a numeric field and a group field
numeric_field_id = None
group_field_id = None

if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    # Try to automatically choose a numeric field (float or int columns)
    numeric_candidates = [
        col for col in df.columns 
        if np.issubdtype(df[col].dropna().dtype, np.number)
    ]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    # Try to choose a group field (non-numeric)
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break

    print(f"Numeric field chosen: {numeric_field_id}")
    print(f"Group field chosen: {group_field_id}")

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].quantile(0.75) # 75th percentile as example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization (Z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouped statistics
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found to analyze.")
else:
    print("No example RecordSet data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if numeric field chosen
if example_record_set_id and example_record_set_id in dataframes and numeric_field_id is not None:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Boxplot by group (if available)
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we:
- Loaded Croissant metadata and record sets for the FAIR^2 dataset.
- Reviewed record sets, their fields, and `@id` assignments.
- Loaded data from a selected record set into a DataFrame and performed initial exploratory data analysis, including filtering, normalization, and grouped aggregation.
- Visualized distributions and relationships in the data.

Refer to [FAIR^2 documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for more information on the data and field semantics.
